- Plant Of Action
    - Use CNN On Fashion MNIST DATA
        1) Basic CNN Architecture
        2) Optimize CNN, hyperparameter tuning

- What is CNN?
    - It is a type of neural network which give best performence on grid based data (images,videos).
    - Lets take example of image classification to identify is cat or not.
        - The architecture is divide into two parts:
            1) Feature Extraction:
                - Extract key feature like tail,moustache
                - Perfrom two operations hera
                    1) Convolution Operation(Convolution Layer)
                        - We have filters and apply these filters on images
                        - The goal is to find out low level fatures like edges
                    2) Pooling Operation(Pooling Layer)
                        - Reduction of tensor size (image size)
            2) Classification:
                - Use ANN Here

    - Overview:
        - (convolution layer pooling layer) ---> Tensor  ---> Flatten ---> 1D ---> ANN

In [ ]:
%pip install optuna

In [46]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader

In [47]:
torch.manual_seed(42)

In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [49]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [50]:
dataset_path = "/content/drive/MyDrive/DataSets/fashion-mnist_train.csv"

In [51]:
df = pd.read_csv(dataset_path)
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [52]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [53]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [54]:
X_train = X_train/255.0
X_test = X_test/255.0

In [55]:
# creatd customdatset class
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28)
        self.labels = torch.tensor(labels,dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [56]:
train_datset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [57]:
train_loader = DataLoader(train_datset,batch_size=32,shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [58]:
class MyCNN(nn.Module):
    def __init__(self,input_features):
        super().__init__()
        # feature extraction stage
        self.features = nn.Sequential(
            nn.Conv2d(input_features,32,kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(32,64,kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2,stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            nn.Linear(64,10)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [59]:
learning_rate = 0.1
epochs = 100

In [60]:
model = MyCNN(1)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)

In [61]:
# training loop
for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        # foward pass
        outputs = model(batch_features)
        # calculate loss
        loss = criterion(outputs,batch_labels)
        # back passs
        optimizer.zero_grad()
        loss.backward()
        # update grads
        optimizer.step()
        total_epoch_loss += loss.item()
    avg_loss = total_epoch_loss/len(train_loader)
    print(f"Epoch: {epoch+1},Loss: {avg_loss}")


Epoch: 1,Loss: 0.5835890574405591
Epoch: 2,Loss: 0.39814127804835636
Epoch: 3,Loss: 0.3510398185700178
Epoch: 4,Loss: 0.30598196389277776
Epoch: 5,Loss: 0.2862500094771385
Epoch: 6,Loss: 0.2604936137807866
Epoch: 7,Loss: 0.24294451874742906
Epoch: 8,Loss: 0.2271125832547744
Epoch: 9,Loss: 0.21364911056930821
Epoch: 10,Loss: 0.2076318084994952
Epoch: 11,Loss: 0.19978469093268117
Epoch: 12,Loss: 0.18642226147434363
Epoch: 13,Loss: 0.18037204304834206
Epoch: 14,Loss: 0.17208401302030932
Epoch: 15,Loss: 0.16291061999214193
Epoch: 16,Loss: 0.16043685874862906
Epoch: 17,Loss: 0.1536688734240209
Epoch: 18,Loss: 0.14700029952529198
Epoch: 19,Loss: 0.14045093209901824
Epoch: 20,Loss: 0.13677541909087448
Epoch: 21,Loss: 0.13775752221955917
Epoch: 22,Loss: 0.13585027646692469
Epoch: 23,Loss: 0.1275234704776667
Epoch: 24,Loss: 0.12236110795134057
Epoch: 25,Loss: 0.12227027813438326
Epoch: 26,Loss: 0.11849862146556067
Epoch: 27,Loss: 0.11660511823270159
Epoch: 28,Loss: 0.1162858048522224
Epoch: 29,

In [62]:
model.eval()

MyCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [63]:
# evaluation on test data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.91625


In [64]:
# evaluation on training data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.9955833333333334
